In [20]:
import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sentence_transformers import SentenceTransformer
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
import dagshub
import numpy as np

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import pandas as pd

# Paste the path you copied from the folder menu here
file_path = '/content/drive/MyDrive/Sentiment analysis /cleaned_data3.csv'

# Load the data into the 'df' variable
df = pd.read_csv(file_path)
df.dropna(inplace = True)
df["Sentiment"] = df["Sentiment"].astype("int8")

In [18]:
dagshub.init(repo_owner= 'vaibhav-patel01' , repo_name= 'YT_Comments_Analysis_chrome_extension' , mlflow= True)

mlflow.set_tracking_uri("https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow")
mlflow.set_experiment("finding_best_feature_engineering_algo")

Initialized MLflow to track repo "vaibhav-patel01/YT_Comments_Analysis_chrome_extension"

Repository vaibhav-patel01/YT_Comments_Analysis_chrome_extension initialized!

<Experiment: artifact_location='mlflow-artifacts:/cf5821c3c16a404c81ff005e5f1f60f0', creation_time=1783399519152, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783399519152, lifecycle_stage='active', name='finding_best_feature_engineering_algo', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [19]:
mlflow.set_experiment("selecting best algo")

<Experiment: artifact_location='mlflow-artifacts:/8c2385d59dbe485781b9535aaf601075', creation_time=1783570722396, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1783570722396, lifecycle_stage='active', name='selecting best algo', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [21]:
comments = df["CommentText"].tolist()
model = SentenceTransformer("all-MiniLM-L6-v2")

# GPU Encoding: Fast, simple, and has a native progress bar
comments_embeddings = model.encode(
    comments,
    batch_size=256,          # Bump this up from 64 to 256 to feed the GPU faster
    show_progress_bar=True,  # This works perfectly here
    convert_to_numpy=True
).astype(np.float32)

# Save to disk

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3710 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
np.save("/content/drive/MyDrive/Sentiment analysis /comments_embeddings4.npy", comments_embeddings)

In [13]:
loaded_embeddings = np.load("/content/drive/MyDrive/Sentiment analysis /comments_embeddings1.npy")

In [14]:
print(df.shape)
print(loaded_embeddings.shape)

(949621, 2)
(971460, 384)


In [ ]:
warnings.filterwarnings("ignore", category=UserWarning)

# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['Sentiment'] = df['Sentiment'].map({-1: 0, 0: 1, 1: 2})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['Sentiment'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3)  # Trigram
max_features = 5000   # Set max_features to 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['CommentText'])
y = df['Sentiment']

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Function to log results in MLflow
# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: TF-IDF Trigrams, model={model_name}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")

# Step 6: Optuna objective function for LightGBM
def objective_svm(trial):
    C = trial.suggest_float('C', 1e-3, 1e2, log=True)
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
    gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

    # degree only matters for 'poly' kernel, but Optuna still needs to suggest it
    # so the search space stays consistent across trials
    degree = trial.suggest_int('degree', 2, 5) if kernel == 'poly' else 3

    model = SVC(C=C, kernel=kernel, gamma=gamma, degree=degree, random_state=42)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))

# Step 7: Run Optuna for SVM, log the best model only
def run_optuna_experiment_svm():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_svm, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = SVC(
        C=best_params['C'],
        kernel=best_params['kernel'],
        gamma=best_params['gamma'],
        degree=best_params.get('degree', 3),
        random_state=42
    )

    # Log the best model with MLflow, passing the algo_name as "SVM"
    log_mlflow("SVM", best_model, X_train, X_test, y_train, y_test)

# Run the experiment for SVM
run_optuna_experiment_svm()